In [ ]:
%cd ../..
import os
import polars as pl
import numpy as np
import random
import itertools
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, BatchSampler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from tqdm import tqdm

from evaluation import *

random.seed(4)
np.random.seed(4)
torch.manual_seed(4)

In [ ]:
embeddings_path = "/scratch/scratch1/embeddings/RSNA-PE/demo"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
def get_embedding(series_uid, slice_idx):
    """Loads a specific embedding tensor from a file."""
    x = torch.load(os.path.join(embeddings_path, f"{series_uid}.pth"), mmap=True)
    return x["cls"][slice_idx]

class RSNAPE(Dataset):
    """Custom PyTorch Dataset for the RSNA PE data."""
    def __init__(self, labels_df):
        self.labels_df = labels_df

    def __len__(self):
        return len(self.labels_df)
    
    def get_labels(self):
        return self.labels_df["has_pe"].to_list()
    
    def __getitem__(self, idx):
        series_uid, slice_idx, has_pe, _ = self.labels_df.row(idx)
        embedding = get_embedding(series_uid, slice_idx)
        label = torch.tensor(has_pe, dtype=torch.float32)
        return embedding, label
    
class StratifiedBatchSampler(BatchSampler):
    """A custom sampler to ensure each batch has a 50/50 class distribution."""
    def __init__(self, labels, batch_size, drop_last=False):
        self.labels = np.array(labels)
        self.batch_size = batch_size
        self.drop_last = drop_last

        assert batch_size % 2 == 0, "Batch size must be even for equal stratification."
        self.half_bs = self.batch_size // 2

        self.pos_indices = np.where(self.labels)[0].tolist()
        self.neg_indices = np.where(~self.labels)[0].tolist()
        
        self.min_class_len = min(len(self.pos_indices), len(self.neg_indices))
        self.num_batches = self.min_class_len // self.half_bs

    def __iter__(self):
        pos = np.random.permutation(self.pos_indices).tolist()[:self.min_class_len]
        neg = np.random.permutation(self.neg_indices).tolist()[:self.min_class_len]

        for i in range(0, self.min_class_len, self.half_bs):
            if i + self.half_bs > self.min_class_len and self.drop_last:
                break
            batch = pos[i:i+self.half_bs] + neg[i:i+self.half_bs]
            np.random.shuffle(batch)
            yield batch

    def __len__(self):
        return self.num_batches

In [ ]:
def get_predictions(model, dataloader, device):
    all_labels = []
    all_predictions = []
    model.eval()
    with torch.no_grad():
        for embeddings, labels in tqdm(dataloader, total=len(dataloader)):
            embeddings = embeddings.to(device)
            labels = labels.to(device)

            predictions = model(embeddings).flatten()
            all_labels.append(labels.cpu())
            all_predictions.append(predictions.cpu())

    all_predictions = torch.cat(all_predictions, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return all_labels, all_predictions

def train_classifier(
    model,
    optimizer,
    loss_fn,
    train_dataloader,
    val_dataloader,
    num_epochs,
    device,
    select_criteria="f1",
    threshold=0.5,
):
    """Main function to train and validate the classifier."""
    assert select_criteria in ["loss", "rocauc", "precision", "recall", "f1"]

    history = {
        "train_loss": [], "train_rocauc": [], "train_precision": [], "train_recall": [], "train_f1": [],
        "val_loss": [], "val_rocauc": [], "val_precision": [], "val_recall": [], "val_f1": [],
    }
    
    best_val_metric = -1 if select_criteria != "loss" else float("inf")
    best_model_state = model.state_dict()

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_all_labels = []
        train_all_predictions = []

        for embeddings, labels in tqdm(train_dataloader, desc=f"Train {epoch+1}/{num_epochs}", leave=False):
            embeddings, labels = embeddings.to(device), labels.to(device)
            predictions = model(embeddings).flatten()
            loss = loss_fn(predictions, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_all_labels.append(labels.detach().cpu())
            train_all_predictions.append(predictions.detach().cpu())

        train_labels_cat = torch.cat(train_all_labels)
        train_predictions_cat = torch.cat(train_all_predictions)
        train_probabilities = torch.sigmoid(train_predictions_cat)
        train_binary_predictions = (train_probabilities >= threshold).long()
        
        history["train_loss"].append(train_loss / len(train_dataloader))
        history["train_rocauc"].append(roc_auc_score(train_labels_cat, train_probabilities))
        history["train_precision"].append(precision_score(train_labels_cat, train_binary_predictions, zero_division=0))
        history["train_recall"].append(recall_score(train_labels_cat, train_binary_predictions, zero_division=0))
        history["train_f1"].append(f1_score(train_labels_cat, train_binary_predictions, zero_division=0))

        model.eval()
        val_loss = 0.0
        val_all_labels = []
        val_all_predictions = []
        with torch.no_grad():
            for embeddings, labels in tqdm(val_dataloader, desc=f"Valid {epoch+1}/{num_epochs}", leave=False):
                embeddings, labels = embeddings.to(device), labels.to(device)
                predictions = model(embeddings).flatten()
                loss = loss_fn(predictions, labels)
                val_loss += loss.item()
                val_all_labels.append(labels.detach().cpu())
                val_all_predictions.append(predictions.detach().cpu())

        val_labels_cat = torch.cat(val_all_labels)
        val_predictions_cat = torch.cat(val_all_predictions)
        val_probabilities = torch.sigmoid(val_predictions_cat)
        val_binary_predictions = (val_probabilities >= threshold).long()

        history["val_loss"].append(val_loss / len(val_dataloader))
        history["val_rocauc"].append(roc_auc_score(val_labels_cat, val_probabilities))
        history["val_precision"].append(precision_score(val_labels_cat, val_binary_predictions, zero_division=0))
        history["val_recall"].append(recall_score(val_labels_cat, val_binary_predictions, zero_division=0))
        history["val_f1"].append(f1_score(val_labels_cat, val_binary_predictions, zero_division=0))

        current_metric = history[f"val_{select_criteria}"][-1]
        if (select_criteria == "loss" and current_metric < best_val_metric) or \
           (select_criteria != "loss" and current_metric > best_val_metric):
            best_val_metric = current_metric
            best_model_state = model.state_dict()
            
    history["state_dict"] = best_model_state
    return history

In [ ]:
print("Loading initial data...")
labels_df = pl.read_csv(os.path.join(embeddings_path, "labels.csv"))
labels_df = labels_df.filter((pl.col("num_img_pe") == 0) | (pl.col("num_img_pe") > 4))
print(f"Full dataset has {len(labels_df)} samples.")

In [ ]:
train_size = 0.8
series_uids = labels_df["series_uid"].unique().to_list()
random.shuffle(series_uids)
num_train_samples = int(train_size * len(series_uids))
train_series_uids = series_uids[:num_train_samples]
val_series_uids = series_uids[num_train_samples:]
train_df_full = labels_df.filter(pl.col("series_uid").is_in(train_series_uids))
val_df_full = labels_df.filter(pl.col("series_uid").is_in(val_series_uids))

print(f"Training samples: {len(train_df_full)}, Validation samples: {len(val_df_full)}")

In [ ]:
param_grid = {
    'learning_rate': [1e-3, 1e-4],
    'hidden_dim': [64, 128],
    'dropout': [0.0, 0.5],
    'weight_decay': [1e-2],
    'batch_size': [512]
}

keys, values = zip(*param_grid.items())
param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

grid_search_results = []
best_rocauc = -1
best_params = None

print(f"Searching {len(param_combinations)} hyperparameter combinations...")
for i, params in enumerate(param_combinations):
    print(f"\nCombination {i+1}/{len(param_combinations)}: {params}")
    
    train_dataset = RSNAPE(train_df_full)
    val_dataset = RSNAPE(val_df_full)
    train_sampler = StratifiedBatchSampler(train_dataset.get_labels(), params['batch_size'], drop_last=True)
    train_dataloader = DataLoader(train_dataset, batch_sampler=train_sampler)
    val_dataloader = DataLoader(val_dataset, batch_size=params['batch_size'], shuffle=False)

    model = nn.Sequential(
        nn.Linear(768, params['hidden_dim']), nn.ReLU(), nn.Dropout(params['dropout']), nn.Linear(params['hidden_dim'], 1)
    ).to(DEVICE)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])

    output = train_classifier(
        model, optimizer, loss_fn, train_dataloader, val_dataloader,
        num_epochs=5, device=DEVICE, select_criteria="f1"
    )
    
    max_val_rocauc = max(output['val_rocauc'])
    params['best_val_rocauc'] = max_val_rocauc
    grid_search_results.append(params)
    
    if max_val_rocauc > best_rocauc:
        best_rocauc = max_val_rocauc
        best_params = params

print("\n--- Grid Search Complete ---")
print(f"Best Validation ROC UAC: {best_rocauc:.4f}")
print(f"Best Hyperparameters: {best_params}")

results_df = pd.DataFrame(grid_search_results)
print("\nFull Grid Search Results:")
print(results_df.sort_values(by='best_val_rocauc', ascending=False).to_string())

In [ ]:
data_ratios = [0.2, 0.4, 0.6, 0.8, 1.0]
data_size_results = []
EMBED_DIM = 768
hidden_dim = 64
batch_size = 512
learning_rate = 1e-3
weight_decay = 1e-2
dropout = 0.5

for ratio in data_ratios:
    print(f"\n--- Training with {ratio*100:.0f}% of the data ---")
    
    num_select_samples = int(ratio * len(series_uids))
    current_series_uids = series_uids[:num_select_samples]
    num_train_subset = int(train_size * len(current_series_uids))
    train_uids_subset = current_series_uids[:num_train_subset]
    train_df_subset = labels_df.filter(pl.col("series_uid").is_in(train_uids_subset))
    val_df_subset = val_df_full
    
    print(f"Training samples: {len(train_df_subset)}, Validation samples: {len(val_df_subset)}")

    train_dataset = RSNAPE(train_df_subset)
    val_dataset = RSNAPE(val_df_subset)
    train_sampler = StratifiedBatchSampler(train_dataset.get_labels(), batch_size, drop_last=True)
    train_dataloader = DataLoader(train_dataset, batch_sampler=train_sampler)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = nn.Sequential(
        nn.Linear(EMBED_DIM, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1)
    ).to(DEVICE)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    output = train_classifier(
        model, optimizer, loss_fn, train_dataloader, val_dataloader,
        num_epochs=10, device=DEVICE, select_criteria="f1"
    )
    
    final_val_f1 = max(output['val_f1'])
    data_size_results.append({'data_ratio': ratio, 'num_train_samples': len(train_df_subset), 'val_f1': final_val_f1})
    print(f"Validation F1 Score for {ratio*100:.0f}% data: {final_val_f1:.4f}")
    
print("\n--- Data Size Testing Complete ---")
data_size_df = pd.DataFrame(data_size_results)
print(data_size_df.to_string(index=False))

plt.figure()
plt.plot(data_size_df['num_train_samples'], data_size_df['val_f1'], marker='o')
plt.title('Model Performance vs. Training Data Size')
plt.xlabel('Number of Training Samples')
plt.ylabel('Best Validation F1 Score')
plt.grid(True)
plt.show()

In [ ]:
num_epochs = 5
EMBED_DIM = 768
hidden_dim = 128
batch_size = 512
learning_rate = 1e-3
weight_decay = 1e-2
dropout = 0.0

train_dataset = RSNAPE(train_df_full)
val_dataset = RSNAPE(val_df_full)
train_sampler = StratifiedBatchSampler(train_dataset.get_labels(), batch_size, drop_last=True)
train_dataloader = DataLoader(train_dataset, batch_sampler=train_sampler)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

model = nn.Sequential(
    nn.Linear(EMBED_DIM, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1)
).to(DEVICE)
loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

output = train_classifier(
    model, optimizer, loss_fn, train_dataloader, val_dataloader,
    num_epochs=num_epochs, device=DEVICE, select_criteria="rocauc"
)


In [ ]:
plot_train_curves(output["train_rocauc"], output["val_rocauc"], "ROC AUC")

In [ ]:
all_labels, all_predictions = get_predictions(model, val_dataloader, DEVICE)
all_predictions = all_predictions > 0.0

plot_confusion_matrix(all_labels, all_predictions, False)